# EB-NeRD — Real Codabench Test Submission

Generates the actual Codabench submission for EB-NeRD/RecSys 2024
(`codabench.org/competitions/2469`) against `ebnerd_testset` -- the real,
blind held-out population, downloaded separately as `ebnerd_testset.zip`
(not part of `data/raw`'s tracked datasets, gitignored). Same reasoning as
`mind_large_test_submission.ipynb` for why this is a standalone notebook
rather than a sixth track through `build_pipeline.ipynb`/`bm25_retrieval.ipynb`/
`embedding_retrieval.ipynb`/`evaluation_harness.ipynb`: `test/behaviors.parquet`
has no `article_ids_clicked` column at all -- a genuine blind test, not a
held-out split of already-labeled data.

**No Kaggle step needed this time** (unlike MIND's real test set): verified
directly that `ebnerd_testset/articles.parquet`'s 125,541 articles are
*exactly* `ebnerd_large`'s existing catalog -- same IDs, zero missing, zero
extra. EB-NeRD shares one fixed article catalog across train/validation/test
(unlike MIND, whose real test set introduces genuinely new articles).
`ebnerd_large`'s already-computed `article_embeddings.parquet` is reused
directly.

Run top-to-bottom to rebuild
`submissions/ebnerd_testset/ebnerd_testset_{method}_predictions.zip`.

Of `test/behaviors.parquet`'s 13,536,710 total rows, 13,336,710 are marked
`is_beyond_accuracy=False` (standard accuracy-focused submission) and
200,000 `is_beyond_accuracy=True` (RecSys 2024's separate diversity-focused
track). **The beyond-accuracy rows are excluded from the scored population,
not just carried through as a flag**: direct inspection found all 200,000
of them share the single literal `impression_id=0` -- a sentinel, not a
real per-impression ID (each row has its own genuine `user_id`/
`session_id`, but the same placeholder impression ID), which breaks the
one-line-per-impression submission format and the uniqueness this
pipeline's format relies on. `is_beyond_accuracy` is parsed and written to
`data/processed/ebnerd_testset/behaviors.parquet` in full for transparency,
but `behaviors_final` (used for scoring and `predictions.txt`) filters
these rows out; implementing the beyond-accuracy track itself (a different
submission format entirely) is out of scope for what's been built so far.

## Setup

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import zipfile

import numpy as np
import polars as pl

from cs4406m26_assignment1c1.bm25 import tokenize, build_index, get_scores
from cs4406m26_assignment1c1.embeddings import mean_pool, cosine_similarity_subset, normalize_rows


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
RAW_DIR = ROOT / "ebnerd_testset"
OUT_DIR = ROOT / "data" / "processed" / "ebnerd_testset"
SUBMISSIONS_DIR = ROOT / "submissions" / "ebnerd_testset"
PROGRESS_LOG = ROOT / "build_progress.log"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

# PREFIX namespaces this population's own users/impressions/sessions.
# ARTICLE_PREFIX is deliberately ebnerd_large's own prefix, not a new one --
# ebnerd_testset shares ebnerd_large's exact article catalog (verified in
# the next cell), so article references here point directly at
# ebnerd_large's already-embedded articles rather than a re-namespaced copy.
PREFIX = "ebnerd_testset_"
ARTICLE_PREFIX = "ebnerd_large_"
RECENT_N_CLICKS = 20
METHODS = ["embedding", "bm25"]


def log_progress(message: str) -> None:
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] {message}\n")
        f.flush()


if not (RAW_DIR / "articles.parquet").exists():
    raise FileNotFoundError(
        f"missing {RAW_DIR / 'articles.parquet'} -- unzip ebnerd_testset.zip at the repo root first "
        "(it must extract to ./ebnerd_testset/)."
    )

log_progress("ebnerd_testset_submission started")
{"raw_dir": str(RAW_DIR), "out_dir": str(OUT_DIR), "submissions_dir": str(SUBMISSIONS_DIR)}

{'raw_dir': 'C:\\Users\\HP\\cs4406m26-assignment1c1\\ebnerd_testset',
 'out_dir': 'C:\\Users\\HP\\cs4406m26-assignment1c1\\data\\processed\\ebnerd_testset',
 'submissions_dir': 'C:\\Users\\HP\\cs4406m26-assignment1c1\\submissions\\ebnerd_testset'}

## Reuse `ebnerd_large`'s article catalog and embeddings

Loaded directly, not re-parsed from `ebnerd_testset/articles.parquet` --
verified below that the two catalogs are exactly the same set of IDs, so
re-parsing would just reproduce (at real cost: another BM25 build, another
embedding load) the same articles this repo already has.

In [2]:
articles = pl.read_parquet(ROOT / "data" / "processed" / "ebnerd_large" / "articles.parquet")
embeddings_df = pl.read_parquet(ROOT / "data" / "processed" / "ebnerd_large" / "article_embeddings.parquet")

test_raw_ids = set(pl.scan_parquet(RAW_DIR / "articles.parquet").select("article_id").collect()["article_id"].to_list())
existing_raw_ids = set(int(a.removeprefix(ARTICLE_PREFIX)) for a in articles["article_id"].to_list())

log_progress(
    f"ebnerd_testset: {len(test_raw_ids)} test articles, {len(existing_raw_ids)} existing ebnerd_large articles, "
    f"{len(test_raw_ids - existing_raw_ids)} missing from existing catalog"
)
{"test_articles": len(test_raw_ids), "existing_articles": len(existing_raw_ids),
 "identical_catalogs": test_raw_ids == existing_raw_ids}

{'test_articles': 125541,
 'existing_articles': 125541,
 'identical_catalogs': True}

In [3]:
def test_catalogs_identical():
    # Hard invariant this whole notebook's "no Kaggle step needed" design
    # depends on -- fail loudly and immediately if it ever stops holding
    # (e.g. a different ebnerd_testset.zip revision with new articles),
    # rather than silently scoring candidates with no embedding.
    assert test_raw_ids == existing_raw_ids, (
        f"ebnerd_testset's article catalog no longer matches ebnerd_large's -- "
        f"{len(test_raw_ids - existing_raw_ids)} missing, {len(existing_raw_ids - test_raw_ids)} extra. "
        "This notebook assumed full overlap and needs a Kaggle embeddings step for the gap if not."
    )
    assert set(embeddings_df["article_id"].to_list()) == set(articles["article_id"].to_list())


test_catalogs_identical()
print(f"ok: ebnerd_testset's {len(test_raw_ids)} articles exactly match ebnerd_large's existing catalog and embeddings")

ok: ebnerd_testset's 125541 articles exactly match ebnerd_large's existing catalog and embeddings


## Build BM25 index and embedding corpus matrix

Same construction as `mind_large_test_submission.ipynb` (`id_to_idx`/
`corpus_unit` precomputed once, not per impression -- SPEC.md Q4 #9).

In [4]:
texts = (articles["title"].fill_null("") + " " + articles["abstract"].fill_null("")).to_list()
doc_tokens = [tokenize(t) for t in texts]
bm25_index = build_index(articles["article_id"].to_list(), doc_tokens)
title_by_id = dict(zip(articles["article_id"].to_list(), articles["title"].to_list()))

doc_ids = articles["article_id"].to_numpy()
emb_lookup = dict(zip(
    embeddings_df["article_id"].to_list(),
    [np.asarray(v) for v in embeddings_df["embedding"].to_list()],
))
matrix = np.stack([emb_lookup[aid] for aid in doc_ids]).astype(np.float32)
id_to_idx = {aid: i for i, aid in enumerate(doc_ids)}
corpus_unit = normalize_rows(matrix)

log_progress(f"ebnerd_testset: BM25 index ({bm25_index.n_docs} docs) and embedding matrix {matrix.shape} built")
{"bm25_docs": bm25_index.n_docs, "avgdl": bm25_index.avgdl, "embedding_matrix_shape": matrix.shape}

{'bm25_docs': 125541,
 'avgdl': np.float64(24.37731896352586),
 'embedding_matrix_shape': (125541, 768)}

In [5]:
def test_bm25_and_embedding_corpus():
    assert bm25_index.n_docs == articles.height
    assert bm25_index.avgdl > 0
    assert matrix.shape == (articles.height, 768)
    assert not np.isnan(matrix).any()
    assert (doc_ids == articles["article_id"].to_numpy()).all()


test_bm25_and_embedding_corpus()
print("ok: BM25 index and embedding matrix built over ebnerd_large's reused catalog, aligned, no NaNs")

ok: BM25 index and embedding matrix built over ebnerd_large's reused catalog, aligned, no NaNs


## Parse behaviors and history (`test/behaviors.parquet`, `test/history.parquet`)

`test/behaviors.parquet` has no `article_ids_clicked` column at all -- the
real distinguishing feature of a blind test file (see the intro markdown).
Otherwise matches `build_pipeline.ipynb`'s `build_ebnerd_behaviors`/
`build_ebnerd_history` transforms exactly (same column selection as
`ebnerd_large`'s own unified schema), with `article_ids_inview`/
`article_id_sequence` prefixed with `ARTICLE_PREFIX` (pointing at
`ebnerd_large`'s reused catalog) while `user_id`/`impression_id`/
`session_id` get this population's own `PREFIX`. `split` is set to the
constant `"test"` (this whole file *is* the held-out population).

In [6]:
behaviors_raw = pl.read_parquet(
    RAW_DIR / "test" / "behaviors.parquet",
    columns=["impression_id", "impression_time", "article_ids_inview", "user_id", "session_id", "is_beyond_accuracy"],
)

behaviors_all = behaviors_raw.select(
    (pl.lit(PREFIX) + pl.col("impression_id").cast(pl.Utf8)).alias("impression_id"),
    pl.lit("ebnerd_testset").alias("dataset"),
    (pl.lit(PREFIX) + pl.col("user_id").cast(pl.Utf8)).alias("user_id"),
    pl.col("impression_time"),
    pl.col("article_ids_inview").list.eval(pl.lit(ARTICLE_PREFIX) + pl.element().cast(pl.Utf8)).alias("article_ids_inview"),
    (pl.lit(PREFIX) + pl.col("session_id").cast(pl.Utf8)).alias("session_id"),
    pl.lit("test").alias("split"),
    pl.col("is_beyond_accuracy"),
)
behaviors_all.write_parquet(OUT_DIR / "behaviors.parquet")
del behaviors_raw

# is_beyond_accuracy rows all share the single literal impression_id=0
# (a sentinel, not a real per-impression ID -- confirmed by direct
# inspection: 200,000 rows, each with its own genuine user_id/session_id,
# all stamped impression_id=0) and belong to RecSys 2024's separate
# diversity-focused track, which this pipeline doesn't implement (see
# intro markdown). Excluded from the scored population; behaviors.parquet
# on disk still carries the full 13,536,710 rows plus is_beyond_accuracy
# for transparency.
behaviors_final = behaviors_all.filter(~pl.col("is_beyond_accuracy"))

history_raw = pl.read_parquet(
    RAW_DIR / "test" / "history.parquet",
    columns=["user_id", "article_id_fixed", "impression_time_fixed", "read_time_fixed", "scroll_percentage_fixed"],
).unique(subset=["user_id"], keep="first", maintain_order=True)

history_table = history_raw.select(
    (pl.lit(PREFIX) + pl.col("user_id").cast(pl.Utf8)).alias("user_id"),
    pl.lit("ebnerd_testset").alias("dataset"),
    pl.col("article_id_fixed").list.eval(pl.lit(ARTICLE_PREFIX) + pl.element().cast(pl.Utf8)).alias("article_id_sequence"),
    pl.col("impression_time_fixed").alias("timestamp_sequence"),
    pl.col("read_time_fixed").alias("read_time_sequence"),
    pl.col("scroll_percentage_fixed").alias("scroll_percentage_sequence"),
)
history_table.write_parquet(OUT_DIR / "history.parquet")
del history_raw

log_progress(
    f"ebnerd_testset: parsed {behaviors_all.height} impressions ({behaviors_final.height} accuracy-track "
    f"+ {behaviors_all.height - behaviors_final.height} beyond-accuracy), "
    f"{history_table.height} distinct users, from test/behaviors.parquet + test/history.parquet"
)
{"impressions_total": behaviors_all.height, "impressions_scored": behaviors_final.height, "distinct_users": history_table.height}

{'impressions_total': 13536710,
 'impressions_scored': 13336710,
 'distinct_users': 807677}

In [7]:
def test_behaviors_and_history_parsed():
    expected_impressions = pl.scan_parquet(RAW_DIR / "test" / "behaviors.parquet").select(pl.len()).collect().item()
    assert behaviors_all.height == expected_impressions
    n_beyond_accuracy = behaviors_all["is_beyond_accuracy"].sum()
    assert behaviors_final.height == expected_impressions - n_beyond_accuracy
    assert not behaviors_final["is_beyond_accuracy"].any(), "beyond-accuracy rows must be excluded from the scored population"
    assert behaviors_final["impression_id"].n_unique() == behaviors_final.height
    inview_lens = behaviors_final["article_ids_inview"].list.len()
    assert (inview_lens > 0).all(), "every impression must have at least one candidate"
    assert (behaviors_final["dataset"] == "ebnerd_testset").all()
    assert (behaviors_final["split"] == "test").all()

    # schema parity with ebnerd_large's own unified tables, minus the one
    # column that genuinely cannot exist here, plus is_beyond_accuracy
    # (real data, carried through deliberately -- see intro markdown)
    existing_behaviors_schema = pl.read_parquet_schema(ROOT / "data" / "processed" / "ebnerd_large" / "behaviors.parquet")
    expected_cols = set(existing_behaviors_schema.keys()) - {"article_ids_clicked"} | {"is_beyond_accuracy"}
    assert set(behaviors_all.columns) == expected_cols
    assert set(behaviors_final.columns) == expected_cols
    existing_history_schema = pl.read_parquet_schema(ROOT / "data" / "processed" / "ebnerd_large" / "history.parquet")
    assert set(history_table.columns) == set(existing_history_schema.keys())

    # every article referenced in inview/history must resolve to a known,
    # embedded article -- the whole point of verifying catalog identity above
    sample_inview_ids = set(aid for row in behaviors_final["article_ids_inview"].head(1000).to_list() for aid in row)
    assert sample_inview_ids.issubset(set(doc_ids))


test_behaviors_and_history_parsed()
print(
    f"ok: {behaviors_all.height} impressions parsed ({behaviors_final.height} accuracy-track, "
    f"{behaviors_all.height - behaviors_final.height} beyond-accuracy excluded), {history_table.height} users, "
    "schema matches ebnerd_large (minus article_ids_clicked, plus is_beyond_accuracy)"
)

ok: 13536710 impressions parsed (13336710 accuracy-track, 200000 beyond-accuracy excluded), 807677 users, schema matches ebnerd_large (minus article_ids_clicked, plus is_beyond_accuracy)


## score_inview adapters

Same shape as Q4/Q5's adapters, single dataset. `cosine_similarity_subset`
takes the precomputed `corpus_unit`/`id_to_idx` from the earlier cell
(SPEC.md Q4 #9's fix, built in from the start).

In [8]:
history_lookup = dict(zip(history_table["user_id"].to_list(), history_table["article_id_sequence"].to_list()))


def build_user_query_tokens(article_id_sequence, title_lookup: dict, recent_n: int = RECENT_N_CLICKS) -> list[str]:
    recent_ids = list(article_id_sequence)[-recent_n:]
    titles = [title_lookup.get(aid, "") for aid in recent_ids]
    return tokenize(" ".join(t for t in titles if t))


def build_user_query_vector(article_id_sequence, embedding_lookup: dict, recent_n: int = RECENT_N_CLICKS):
    recent_ids = list(article_id_sequence)[-recent_n:]
    return mean_pool(recent_ids, embedding_lookup)


bm25_cache = {"user_id": None, "scores": None}


def bm25_fn(user_id, article_ids_inview):
    if bm25_cache["user_id"] != user_id:
        seq = history_lookup.get(user_id, [])
        query_tokens = build_user_query_tokens(seq, title_by_id)
        bm25_cache["user_id"] = user_id
        bm25_cache["scores"] = get_scores(bm25_index, query_tokens)
    scores = bm25_cache["scores"]
    return {aid: float(scores[id_to_idx[aid]]) for aid in article_ids_inview}


def embedding_fn(user_id, article_ids_inview):
    seq = history_lookup.get(user_id, [])
    query_vector = build_user_query_vector(seq, emb_lookup)
    scored = cosine_similarity_subset(query_vector, corpus_unit, doc_ids, id_to_idx, article_ids_inview)
    return {aid: scored.get(aid, 0.0) for aid in article_ids_inview}


score_inview_adapters = {"bm25": bm25_fn, "embedding": embedding_fn}

In [9]:
def test_score_inview_adapters():
    sample = behaviors_final.row(0, named=True)
    inview = list(sample["article_ids_inview"])
    for method in METHODS:
        scored = score_inview_adapters[method](sample["user_id"], inview)
        assert set(scored) == set(inview)
        assert all(np.isfinite(v) for v in scored.values())

    coldstart_rows = history_table.filter(pl.col("article_id_sequence").list.len() == 0)
    if coldstart_rows.height > 0:
        coldstart_user = coldstart_rows.row(0, named=True)["user_id"]
        bm25_scored = score_inview_adapters["bm25"](coldstart_user, inview)
        assert set(bm25_scored.values()) == {0.0}
        emb_scored = score_inview_adapters["embedding"](coldstart_user, inview)
        assert set(emb_scored.values()) == {0.0}

    fn = score_inview_adapters["bm25"]
    assert fn(sample["user_id"], inview) == fn(sample["user_id"], inview)


test_score_inview_adapters()
print("ok: score_inview adapters have a uniform signature, no NaN/inf, and score cold-start users as an all-zero tie")

ok: score_inview adapters have a uniform signature, no NaN/inf, and score cold-start users as an all-zero tie


## Generate predictions

Per SPEC.md Q5 #2/#4, same format as `generate_predictions.ipynb`: one line
per impression, `{native_impression_id} [{rank_1},...,{rank_n}]`, native ID
recovered as `impression_id`'s trailing token after the last `_` (same
convention as every other EB-NeRD dataset). Row order matches
`test/behaviors.parquet`'s own row order. Zip contains exactly
`predictions.txt` at the root (EB-NeRD's plural filename convention, SPEC.md
Q5 #2 -- inferred from the starter repo, not yet directly confirmed against
Codabench's own guidelines, same caveat as every other EB-NeRD submission
here).

In [10]:
def generate_predictions(method: str) -> Path:
    split_behaviors = behaviors_final.with_row_index("row_idx")
    score_fn = score_inview_adapters[method]

    compute_order = split_behaviors.sort("user_id")
    row_idx_col = compute_order["row_idx"].to_list()
    user_id_col = compute_order["user_id"].to_list()
    inview_col = compute_order["article_ids_inview"].to_list()
    n_rows = len(row_idx_col)
    log_progress(f"ebnerd_testset/{method}: scoring {n_rows} impressions")

    ranks_by_row = {}
    for i, (row_idx, user_id, inview) in enumerate(zip(row_idx_col, user_id_col, inview_col)):
        inview_ids = list(inview)
        scored = score_fn(user_id, inview_ids)
        scores = np.array([scored[aid] for aid in inview_ids])
        ranks = (np.argsort(np.argsort(-scores, kind="stable"), kind="stable") + 1).tolist()
        ranks_by_row[row_idx] = ranks
        if (i + 1) % 200_000 == 0:
            log_progress(f"    ebnerd_testset/{method}: {i + 1}/{n_rows} impressions scored")

    lines = []
    for row_idx, impression_id in zip(split_behaviors["row_idx"].to_list(), split_behaviors["impression_id"].to_list()):
        native_id = impression_id.rsplit("_", 1)[-1]
        ranks_str = "[" + ",".join(str(r) for r in ranks_by_row[row_idx]) + "]"
        lines.append(f"{native_id} {ranks_str}")

    txt_path = SUBMISSIONS_DIR / "predictions.txt"
    # write_text() would translate \n -> \r\n on Windows; write bytes
    # directly so the file matches the reference implementation exactly.
    txt_path.write_bytes(("\n".join(lines) + "\n").encode("utf-8"))

    zip_path = SUBMISSIONS_DIR / f"ebnerd_testset_{method}_predictions.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(txt_path, arcname="predictions.txt")
    txt_path.unlink()
    log_progress(f"ebnerd_testset/{method}: wrote {zip_path.name} ({len(lines)} lines)")
    return zip_path


prediction_zips = {method: generate_predictions(method) for method in METHODS}
log_progress("ebnerd_testset_submission: all zips written")
prediction_zips

{'embedding': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/submissions/ebnerd_testset/ebnerd_testset_embedding_predictions.zip'),
 'bm25': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/submissions/ebnerd_testset/ebnerd_testset_bm25_predictions.zip')}

In [11]:
def test_generate_predictions():
    n_expected = behaviors_final.height
    expected_ids = [iid.rsplit("_", 1)[-1] for iid in behaviors_final["impression_id"].to_list()]
    expected_inview_lens = [len(x) for x in behaviors_final["article_ids_inview"].to_list()]

    for method in METHODS:
        zip_path = prediction_zips[method]
        assert zip_path.exists()
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
            assert names == ["predictions.txt"], f"zip must contain exactly predictions.txt at root, got {names}"
            content = zf.read("predictions.txt").decode("utf-8")

        lines = content.strip("\n").split("\n")
        assert len(lines) == n_expected

        for line, expected_id, expected_len in zip(lines, expected_ids, expected_inview_lens):
            impr_id_str, ranks_str = line.split(" ", 1)
            assert impr_id_str == expected_id, "row order must match test/behaviors.parquet's original row order"
            ranks = [int(r) for r in ranks_str.strip("[]").split(",")]
            assert len(ranks) == expected_len
            assert sorted(ranks) == list(range(1, expected_len + 1)), "ranks must be a permutation starting at 1"


test_generate_predictions()
log_progress("ebnerd_testset_submission completed successfully")
print("ok: predictions.txt round-trips for every method -- correct row count, row order, and valid rank permutations")

ok: predictions.txt round-trips for every method -- correct row count, row order, and valid rank permutations


# Manual Review Complete

Unlike `ebnerd_large`'s/`ebnerd_small`'s submission zips (see
`generate_predictions.ipynb`'s intro / SPEC.md Q5 #3),
`submissions/ebnerd_testset/ebnerd_testset_embedding_predictions.zip`
**is** the real, submittable Codabench population for
`codabench.org/competitions/2469` — upload directly under Participate →
Submit / View Results. Prefer the `embedding` method zip for the actual
submission (Q4 found embeddings ranking better than BM25 on every dataset's
test split); the `bm25` zip is for comparison only. EB-NeRD's
`predictions.txt` filename (plural) is inferred, not yet directly confirmed
against Codabench's own Submission Guidelines page — verify on first
upload.